# RAG MVP: PDF Q&A with Gemini + FAISS

Pipeline: **PDF → text → chunks (with overlap) → Gemini embeddings → FAISS → retrieve top-k → Gemini LLM → answer**

Get a free API key: https://aistudio.google.com/apikey

In [ ]:
%pip install langchain==1.3.6 langchain-community==0.4.2 langchain-google-genai==4.4.0 pypdf==6.10.2 faiss-cpu==1.15.0


In [1]:
# check package varsion
import importlib.metadata

# List the distribution package names
packages = ["langchain", "langchain-community", 
             "langchain-google-genai", 
             "pypdf",  "faiss-cpu"]

for package in packages:
    try:
        version = importlib.metadata.version(package)
        print(f"{package} version: {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package} is not installed in this environment.")

# langchain version: 1.3.6
# langchain-community version: 0.4.2
# langchain-google-genai version: 4.4.0
# pypdf version: 6.10.2
# faiss-cpu version: 1.15.0

langchain version: 1.3.6
langchain-community version: 0.4.2
langchain-google-genai version: 4.4.0
pypdf version: 6.10.2
faiss-cpu version: 1.15.0


## 1. Setup

In [2]:
import os, time, getpass
from pypdf import PdfReader
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI


if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Gemini API key: ")

C:\Users\ash32\AppData\Local\Temp\ipykernel_15900\3543716649.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Gemini API key:  ········


In [4]:
# --- Config ---
PDF_PATH        = "document-loxford-company.pdf"             # change to your PDF
CHUNK_SIZE      = 1000                     # characters per chunk
CHUNK_OVERLAP   = 200                      # characters shared between neighbouring chunks
TOP_K           = 4                        # chunks retrieved per question
EMBED_MODEL     = "models/gemini-embedding-001"
LLM_MODEL       = "gemini-2.5-flash"       # swap if the model is retired


## 2. Read the PDF and extract text

In [6]:
reader = PdfReader(PDF_PATH)
pages = [(i + 1, (p.extract_text() or "").strip()) for i, p in enumerate(reader.pages)]
pages = [(n, t) for n, t in pages if t]   # drop empty pages (scanned pages need OCR)

print(f"Pages with text: {len(pages)} / {len(reader.pages)}")
print(f"Total characters: {sum(len(t) for _, t in pages)}")
print("\nSample:\n", pages[0][1][:800])

Pages with text: 3 / 3
Total characters: 2738

Sample:
 Loxford Technologies – Company Overview 
Loxford Technologies is a global technology solutions provider specializing in 
artificial intelligence, data analytics, and enterprise automation. Founded in 2012, 
the company has grown rapidly into a mid-sized enterprise serving clients across 
North America, Europe, and Asia. 
Loxford Technologies was founded by Daniel Reeves, a former data scientist, and 
Anika Lomax, a software engineer with a background in distributed systems. The 
founders envisioned a company that could bridge the gap between cutting-edge 
AI research and real-world business applications. 
• CEO: Daniel Reeves 
• CTO: Anika Lomax 
The company is headquartered in Austin, Texas, USA, with additional offices in: 
• London, UK 
• Bengaluru, India 
• Berlin, Germany 
As of 2024,


## 3. Chunking with overlap

A fixed-size sliding window. Each chunk starts `CHUNK_SIZE - CHUNK_OVERLAP` characters after the previous one, so text at a boundary appears in both chunks.

In [7]:
def chunk_text(text, size, overlap):
    step = size - overlap
    return [text[i:i + size] for i in range(0, len(text), step) if text[i:i + size].strip()]

docs = []
for page_no, text in pages:
    for j, chunk in enumerate(chunk_text(text, CHUNK_SIZE, CHUNK_OVERLAP)):
        docs.append(Document(page_content=chunk, metadata={"page": page_no, "chunk": j}))

print(f"Chunks created: {len(docs)}")
print("\nChunk 0:\n", docs[0].page_content[:300])
if len(docs) > 1:
    print("\nOverlap check (end of chunk 0 vs start of chunk 1):")
    print("END  :", repr(docs[0].page_content[-CHUNK_OVERLAP:][:100]))
    print("START:", repr(docs[1].page_content[:100]))

Chunks created: 5

Chunk 0:
 Loxford Technologies – Company Overview 
Loxford Technologies is a global technology solutions provider specializing in 
artificial intelligence, data analytics, and enterprise automation. Founded in 2012, 
the company has grown rapidly into a mid-sized enterprise serving clients across 
North Ameri

Overlap check (end of chunk 0 vs start of chunk 1):
END  : ' Loxford Technologies reported an estimated annual revenue of $180 \nmillion, with a year-over-year g'
START: ' Loxford Technologies reported an estimated annual revenue of $180 \nmillion, with a year-over-year g'


In [8]:
# Lets print all the chunks
for doc in docs:
    print(f"--- Page {doc.metadata['page']}, Chunk {doc.metadata['chunk']} ---")
    print(doc.page_content)
    print("\n" + "="*50 + "\n")


--- Page 1, Chunk 0 ---
Loxford Technologies – Company Overview 
Loxford Technologies is a global technology solutions provider specializing in 
artificial intelligence, data analytics, and enterprise automation. Founded in 2012, 
the company has grown rapidly into a mid-sized enterprise serving clients across 
North America, Europe, and Asia. 
Loxford Technologies was founded by Daniel Reeves, a former data scientist, and 
Anika Lomax, a software engineer with a background in distributed systems. The 
founders envisioned a company that could bridge the gap between cutting-edge 
AI research and real-world business applications. 
• CEO: Daniel Reeves 
• CTO: Anika Lomax 
The company is headquartered in Austin, Texas, USA, with additional offices in: 
• London, UK 
• Bengaluru, India 
• Berlin, Germany 
As of 2024, Loxford Technologies reported an estimated annual revenue of $180 
million, with a year-over-year growth rate of approximately 22%. The company 
employs over 850 professionals

## 4. Embed chunks and store in FAISS

In [9]:
embeddings = GoogleGenerativeAIEmbeddings(model=EMBED_MODEL)

# Peek at what an embedding looks like
vec = embeddings.embed_query("hello world")
print(f"Embedding dimension: {len(vec)}")
print("First 5 values:", [round(v, 4) for v in vec[:5]])

Embedding dimension: 3072
First 5 values: [-0.0242, 0.0099, 0.0075, -0.0673, 0.0001]


In [10]:
# Batch to stay within free-tier rate limits
BATCH, PAUSE = 50, 2
vectorstore = None
for i in range(0, len(docs), BATCH):
    batch = docs[i:i + BATCH]
    if vectorstore is None:
        vectorstore = FAISS.from_documents(batch, embeddings)
    else:
        vectorstore.add_documents(batch)
    print(f"Indexed {min(i + BATCH, len(docs))}/{len(docs)}")
    time.sleep(PAUSE)

print("Vectors in index:", vectorstore.index.ntotal)

Indexed 5/5
Vectors in index: 5


## 5. Retrieval from Vector DB (no LLM yet)

In [13]:
def show_retrieval(question, k=TOP_K):
    results = vectorstore.similarity_search_with_score(question, k=k)  # score = L2 distance, lower is closer
    for doc, score in results:
        print(f"[page {doc.metadata['page']} | chunk {doc.metadata['chunk']} | distance {score:.3f}]")
        print(doc.page_content[:250].replace("\n", " "), "...\n")

show_retrieval("What is the annual revenue ?")

[page 1 | chunk 1 | distance 0.538]
 Loxford Technologies reported an estimated annual revenue of $180  million, with a year-over-year growth rate of approximately 22%. The company  employs over 850 professionals, including engineers, data scientists, and  consultants.  . Loxford Techn ...

[page 1 | chunk 0 | distance 0.641]
Loxford Technologies – Company Overview  Loxford Technologies is a global technology solutions provider specializing in  artificial intelligence, data analytics, and enterprise automation. Founded in 2012,  the company has grown rapidly into a mid-si ...

[page 2 | chunk 1 | distance 0.728]
 Retail & E-commerce  • Manufacturing  • Education ...

[page 3 | chunk 0 | distance 0.750]
Recent Developments   In 2026, the company launched its first Agentic AI platform, enabling autonomous  decision-making systems for enterprise clients. It has also partnered with several  Fortune 500 companies to deploy large-scale AI solutions.  Com ...



## 6. Generation with Gemini

In [16]:
llm = ChatGoogleGenerativeAI(model=LLM_MODEL, temperature=0)

system_prompt = """
     Answer the question using only the context.
     If the answer is not in the context, say you don't know.
"""

human_prompt = """
     Answer the question using only the context below.
     If the answer is not in the context, say you don't know.
     
     Context:
     {context}

     Question:
     {question}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human",human_prompt),
])

def ask(question, k=TOP_K, show_sources=True):
    retrieved = vectorstore.similarity_search(question, k=k)
    context = "\n\n---\n\n".join(
        f"(page {d.metadata['page']}) {d.page_content}" for d in retrieved
    )
    answer = (prompt | llm).invoke({"context": context, "question": question})
    print("Q:", question)
    print("A:", answer.text)
    if show_sources:
        print("Sources: pages", sorted({d.metadata["page"] for d in retrieved}))
    return answer

In [17]:
ask("What is the annual revenue ?")

Q: What is the annual revenue ?
A: The annual revenue is $180 million.
Sources: pages [1, 2, 3]


AIMessage(content='The annual revenue is $180 million.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0f55a-8e91-76b2-ac27-84dac12edd00-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 605, 'output_tokens': 153, 'total_tokens': 758, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 143}})

In [18]:
# Out-of-scope question: should answer "I don't know"

ask("Who won the FIFA World Cup in 2018?")

Q: Who won the FIFA World Cup in 2018?
A: I don't know.
Sources: pages [1, 2, 3]


AIMessage(content="I don't know.", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0f55a-a6e1-7352-b6ce-c8a7e2125e74-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 664, 'output_tokens': 269, 'total_tokens': 933, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 263}})

## 7. Optional: save and reload the index

In [19]:
vectorstore.save_local("faiss_index")

# Later / in another session:
# vectorstore = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)

## Demo ideas for students

- Change `CHUNK_SIZE` / `CHUNK_OVERLAP` (e.g. 200/0 vs 1000/200) and compare retrieval results.
- Change `TOP_K` and watch answer quality and hallucination.
- Ask something not in the PDF.
- Compare `show_retrieval()` output with the final answer to see what the LLM actually saw.